In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
from sklearn.manifold import TSNE

In [ ]:
d = {
    "even": {
        0: 1,
        1: 0,
        2: 1,
        3: 0,
        4: 1,
        5: 0,
        6: 1,
        7: 0,
        8: 1,
        9: 0
    },
    "greater_4": {
        0: 0,
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        5: 1, 
        6: 1,
        7: 1,
        8: 1,
        9: 1
    },
    "multiple_3": {
        0: 1,
        1: 0,
        2: 0,
        3: 1, 
        4: 0,
        5: 0,
        6: 1,
        7: 0,
        8: 0,
        9: 1
    }, 
    "prime": {
        0: 0,
        1: 1,
        2: 1,
        3: 1,
        4: 0,
        5: 1,
        6: 0,
        7: 1,
        8: 0,
        9: 0
    },
    "not_outlier": {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 1,
        5: 1, 
        6: 1,
        7: 0,
        8: 0,
        9: 0
    }
}

In [ ]:
count = {i: 0 for i in range(10)}
for k in d:
    for i in range(10):
        if d[k][i] == 1:
            count[i] += 1

In [ ]:
count

In [ ]:
from sklearn.datasets import fetch_openml

# Scarica il dataset MNIST
mnist = fetch_openml('mnist_784', version=1)

In [ ]:
df = mnist["data"]

In [ ]:
df["target"] = mnist["target"]

In [ ]:
df["target"] = df["target"].astype(int)

In [ ]:
df["task1"] = df["target"].apply(lambda x : d["even"][x])
df["task2"] = df["target"].apply(lambda x : d["greater_4"][x])
df["task3"] = df["target"].apply(lambda x : d["multiple_3"][x])
df["task4"] = df["target"].apply(lambda x : d["prime"][x])
df["task5"] = df["target"].apply(lambda x : d["not_outlier"][x])

In [ ]:
for i in range(1, 785):
    df[f"pixel{i}"] = df[f"pixel{i}"] / 255

In [ ]:
df = df.sample(frac=1).reset_index(drop=True)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.1667, stratify=df['target'])
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

In [ ]:
df.to_csv(os.path.join(root, "datasets", "mnist.csv"), index=False)
train_df.to_csv(os.path.join(root, "datasets", "mnist_train.csv"), index=False)
test_df.to_csv(os.path.join(root, "datasets", "mnist_test.csv"), index=False)

In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import umap

In [ ]:
root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

df = pd.read_csv(os.path.join(root, "datasets", "mnist.csv"))
train_df = pd.read_csv(os.path.join(root, "datasets", "mnist_train.csv"))
test_df = pd.read_csv(os.path.join(root, "datasets", "mnist_test.csv"))

In [ ]:
umap_model = umap.UMAP(n_components=30)
umap_model.fit(train_df.iloc[:,:784].values)

In [ ]:
X_train_red = umap_model.transform(train_df.iloc[:,:784].values)
X_test_red = umap_model.transform(test_df.iloc[:,:784].values)

In [ ]:
def red(X, df):
    df_red = pd.DataFrame(X, columns=[f"feat{i}" for i in range(X.shape[1])])
    df_red["target"] = list(df["target"])
    for i in range(1,6):
        df_red[f"task{i}"] = list(df[f"task{i}"])
    return df_red

In [ ]:
df_train_red = red(X_train_red, train_df)
df_test_red = red(X_test_red, test_df)
df_train_red.to_csv(os.path.join(root, "datasets", "mnist_red30_train.csv"), index=False)
df_test_red.to_csv(os.path.join(root, "datasets", "mnist_red30_test.csv"), index=False)

In [1]:
from dataset_utils import *
import os
import pandas as pd

In [2]:
root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

In [3]:
df_train_red = pd.read_csv(os.path.join(root, "datasets", "mnist_red30_train.csv"))
df_test_red = pd.read_csv(os.path.join(root, "datasets", "mnist_red30_test.csv"))

In [4]:
set1 = [0,2,4,6,8]
set2 = [1,3,5,7,9]

In [5]:
dataset = "mnist_red30"

In [ ]:
exps_train = make_exp_incremental_train(root, dataset, df_train_red, set1, set2)
exps_test = make_exp_incremental_test(root, dataset, df_test_red)

In [ ]:
exps_sml_train = make_exp_sml_train(root, dataset, df_train_red)
exps_sml_test = make_exp_sml_test(root, dataset, df_test_red)

In [ ]:
check_distr(exps_train)
check_distr(exps_test)
check_distr(exps_sml_train)
check_distr(exps_sml_test)

In [ ]:
exps_sml_train_3 = make_exp_sml_train(root, dataset, df_train_red, n_split=3, suffix="_3")
exps_sml_test_3 = make_exp_sml_test(root, dataset, df_test_red, n_split=3, suffix="_3")

In [8]:
exps_sml_train_2 = make_exp_sml_train(
    root,
    dataset,
    df_train_red,
    tasks_perc=[0.3, 0.7],
    suffix="_2"
)
exps_sml_test_2 = make_exp_sml_test(root, dataset, df_test_red, n_split=2, suffix="_2")

(1, 2) ACCEPTED
(5, 3) ACCEPTED
(1, 3) ACCEPTED
(2, 3) ACCEPTED
(2, 1) ACCEPTED
(5, 2) ACCEPTED
(1, 3) ACCEPTED
(5, 3) ACCEPTED
(5, 1) ACCEPTED
(4, 1) ACCEPTED
